# 11. AIND Units NWB Export

Build an `AINDUnitsNWBScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDUnitsNWBTask` for each coordinate. The task itself clones [aind-units-nwb](https://github.com/AllenNeuralDynamics/aind-units-nwb) on first use, patches `utils.py` for two neuroconv API renames (`add_electrodes_info_to_nwbfile` → `add_electrodes_to_nwbfile`, `add_units_table_to_nwbfile` → `_add_units_table_to_nwbfile`), seeds its `data/` with the base NWB + `preprocessed/` / `curated/` / `spikesorted/` / `postprocessed/` from the results-collector layout + dispatch `job_*.json` + a synthetic `ecephys_*` session folder, runs `code/run_capsule.py`, and copies the resulting NWB (with a `units` table) into the single config's `coordinate_output_root` (`obi-output/11_aind_units_nwb/grid_scan/0/`).

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "neuroconv", "pynwb", "hdmf-zarr", "aind-nwb-utils",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 4 packages in 4ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'neuroconv', 'pynwb', 'hdmf-zarr', 'aind-nwb-utils'], returncode=0)

## Build the scan config

In [3]:
nwb_input_path = (
    Path.cwd() / "../../../obi-output/10_aind_ecephys_nwb/grid_scan/0"
).resolve()
collected_output_path = (
    Path.cwd() / "../../../obi-output/07_aind_ephys_results_collector/grid_scan/0"
).resolve()
dispatch_output_path = (
    Path.cwd() / "../../../obi-output/01_aind_ephys_dispatch/grid_scan/0"
).resolve()
for p in (nwb_input_path, collected_output_path, dispatch_output_path):
    assert p.exists(), f"{p} not found. Run earlier notebooks first."

scan_config = obi.AINDUnitsNWBScanConfig(
    initialize=obi.AINDUnitsNWBScanConfig.Initialize(
        nwb_input_path=nwb_input_path,
        collected_output_path=collected_output_path,
        dispatch_output_path=dispatch_output_path,
    ),
)

## Generate the grid scan and run the units-NWB task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/11_aind_units_nwb/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 18:36:17,955] INFO: Cloning https://github.com/AllenNeuralDynamics/aind-units-nwb.git -> /tmp/aind-units-nwb


Cloning into '/tmp/aind-units-nwb'...


[2026-04-29 18:36:19,937] INFO: Patched utils.py for neuroconv API drift


[2026-04-29 18:36:20,107] INFO: Seeded data dir: /tmp/aind-units-nwb/data


[2026-04-29 18:36:20,108] INFO: Running python -u run_capsule.py


[2026-04-29 18:36:22,081] INFO: 

NWB EXPORT UNITS
Running NWB conversion with the following parameters:
Stub test: False
Stub units: 10
NWB backend: hdf5
Found 1 JSON job files
Found 1 processed recordings
Number of NWB files to write: 1
Number of streams to write for each file: 1
	Adding 10 units from block0_None_recording1
Added 1 streams
Writing time: 0.05s
Done writing ../results/session0_block0_recording1.nwb
NWB EXPORT UNITS time: 1.19s



[PosixPath('../../../obi-output/11_aind_units_nwb/grid_scan/0')]

## Inspect the units table

In [5]:
from pynwb import NWBHDF5IO

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

nwb_files = sorted(coord_dir.glob("*.nwb"))
print("NWB files:", [p.name for p in nwb_files])

if nwb_files:
    with NWBHDF5IO(str(nwb_files[0]), "r") as io:
        nwb = io.read()
        units = nwb.units
        if units is None:
            print("No units table written.")
        else:
            print("units columns:", list(units.colnames))
            print("num_units:    ", len(units.id))
            df = units.to_dataframe().reset_index()
            keep = [
                c
                for c in [
                    "id",
                    "unit_name",
                    "ks_unit_id",
                    "device_name",
                    "amplitude",
                    "depth",
                    "extremum_channel_index",
                ]
                if c in df.columns
            ]
            print()
            print(df[keep].to_string(index=False))

coordinate_output_root: ../../../obi-output/11_aind_units_nwb/grid_scan/0
NWB files: ['session0_block0_recording1.nwb']
units columns: ['spike_times', 'electrodes', 'waveform_mean', 'waveform_sd', 'unit_name', 'sync_spike_8', 'device_name', 'sync_spike_2', 'exp_decay', 'sync_spike_4', 'repolarization_slope', 'drift_ptp', 'l_ratio', 'rp_contamination', 'isolation_distance', 'peak_before_to_trough_ratio', 'trough_width', 'decoder_label', 'amplitude_median', 'nn_hit_rate', 'estimated_x', 'firing_rate', 'amplitude_cv_range', 'amplitude_cv_median', 'peak_half_width', 'waveform_baseline_flatness', 'shank', 'original_cluster_id', 'ks_unit_id', 'num_spikes', 'peak_before_to_peak_after_ratio', 'peak_before_width', 'd_prime', 'firing_range', 'velocity_below', 'extremum_channel_index', 'isi_violations_ratio', 'decoder_probability', 'estimated_y', 'isi_violations_count', 'peak_to_trough_duration', 'presence_ratio', 'drift_mad', 'default_qc', 'amplitude', 'depth', 'snr', 'estimated_z', 'num_negativ